In [ ]:
pip install opencv-python torch twilio firebase-admin cloudinary pyserial

Import Liabraries

In [ ]:
import cv2
import time
import torch
from ultralytics import YOLO
from twilio.rest import Client
import firebase_admin
from firebase_admin import credentials, db
import cloudinary
import cloudinary.uploader
from datetime import datetime
import os
import threading
import serial

os.makedirs("Detected_images", exist_ok=True)
os.makedirs("chunk_video", exist_ok=True)
os.makedirs("local_videos", exist_ok=True)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")
print("Libraries imported successfully!")

Importing YOLO model,Twilio, Firebase and Cloudinary

In [ ]:
model = YOLO("runs/detect/train/weights/best.pt").to(device)

account_sid = os.getenv("TWILIO_ACCOUNT_SID")
auth_token = os.getenv("TWILIO_AUTH_TOKEN")
twilio_phone = os.getenv("TWILIO_PHONE")
security_phone = os.getenv("SECURITY_PHONE")
client = Client(account_sid, auth_token)

cred = credentials.Certificate('Credentials.json')
if not firebase_admin._apps:
    firebase_admin.initialize_app(cred, {
        'databaseURL': 'https://secuvision-8b0d0-default-rtdb.asia-southeast1.firebasedatabase.app/'
    })
db_ref = db.reference('shoplifting_detections')

cloudinary.config(
    cloud_name=os.getenv("CLOUDINARY_CLOUD_NAME"),
    api_key=os.getenv("CLOUDINARY_API_KEY"),
    api_secret=os.getenv("CLOUDINARY_API_SECRET")
)
print("Initialized YOLO, Twilio, Firebase, Cloudinary")

Setting up Arduino

In [ ]:
serial_port = "COM3"
baud_rate = 9600
try:
    ser = serial.Serial(serial_port, baud_rate, timeout=1)
    time.sleep(2)
    print(f"Connected to Arduino on {serial_port}")
except serial.SerialException as e:
    print(f"Could not connect to Arduino: {e}")
    ser = None

Importing the video Source

In [ ]:
def initialize_video_source(source):
    cap = cv2.VideoCapture(source)
    if not cap.isOpened():
        print(f"Error: Could not open video source {source}.")
        return None
    print(f"Video source {source} opened successfully!")
    return cap

video_source = input("Enter video source (IP webcam URL like 'http://<ip>:8080/video' or local file path, e.g., 'test_video/1.mp4'): ") or "http://192.168.1.100:8080/video"
cap = initialize_video_source(video_source)
if cap is None:
    exit()

fourcc = cv2.VideoWriter_fourcc(*'mp4v')
chunk_duration = 10
fps = int(cap.get(cv2.CAP_PROP_FPS)) if cap.get(cv2.CAP_PROP_FPS) > 0 else 30
frame_width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH)) or 1280
frame_height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT)) or 720

Processing Video and using threading

In [ ]:
confidence_threshold = 0.7
alert_cooldown = 5
last_alert_time = 0
frame_skip = 2
frame_count = 0

def upload_to_cloudinary(chunk_filename):
    try:
        file_size = os.path.getsize(chunk_filename) / (1024 * 1024)
        print(f"Starting upload of {chunk_filename} ({file_size:.2f} MB)")
        upload_result = cloudinary.uploader.upload(
            chunk_filename,
            resource_type="video",
            public_id=f"detected_clips/{os.path.basename(chunk_filename)}",
            overwrite=True
        )
        print(f"Uploaded to Cloudinary: {upload_result['secure_url']}")
    except Exception as e:
        print(f"Cloudinary upload error for {chunk_filename}: {e}")
    finally:
        if os.path.exists(chunk_filename):
            os.remove(chunk_filename)
            print(f"Deleted local chunk file: {chunk_filename}")

def save_local_video(chunk_filename):
    try:
        local_path = os.path.join("local_videos", os.path.basename(chunk_filename))
        shutil.copy2(chunk_filename, local_path)
        print(f"Saved video locally: {local_path}")
    except Exception as e:
        print(f"Local video save error for {chunk_filename}: {e}")

def save_to_firebase(detection_details):
    try:
        new_detection_ref = db_ref.push(detection_details)
        print(f"Detection stored in Firebase: {new_detection_ref.key}")
    except Exception as e:
        print(f"Firebase error: {e}")

def send_alert(detection_details):
    message_body = f"Shoplifting detected at {time.ctime()}! Probability: {detection_details['confidence']:.2f}"
    try:
        message = client.messages.create(
            body=message_body,
            from_=twilio_phone,
            to=security_phone
        )
        print(f"Alert sent: {message.sid}")
    except Exception as e:
        print(f"Alert error: {e}")

def trigger_led_blink():
    if ser is not None:
        try:
            ser.write(b'B')
            print("Sent blink command to Arduino")
        except serial.SerialException as e:
            print(f"Error sending command to Arduino: {e}")
